# Week 05 — Advanced Model Training
**Course:** Machine Learning & Deep Learning Project  
**Dataset:** MovieLens (Classification Framing)  

---

### SOP Task List (Verbatim):
> **Week 5: Advanced Model Training**
> - [x] Use Advanced Models (e.g., Random Forest, Gradient Boosting, SVM).
> - [x] Perform K-Fold Cross-Validation.
> - [x] Hyperparameter Tuning using GridSearch or RandomSearch.


## Week Objective
We will upgrade from our linear baseline (Logistic Regression) to non-linear tree-based algorithms capable of capturing complex interactions between user preferences and movie popularity. We will train **Random Forest** and **Histogram Gradient Boosting** models. To ensure robustness, we will evaluate using **5-Fold Cross-Validation** and optimize the best model using **Randomized Search CV** for hyperparameter tuning.

*(Note: To execute computationally expensive tuning in a reasonable time, we will perform CV and Tuning on a representative stratified subsample of our data).*


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score

# Robust path resolution
base_parent = ".." if os.path.exists(os.path.join("..", "CSVs")) else "."
def resolve_file(rel_path):
    for candidate in [os.path.join("..", rel_path), rel_path]:
        if os.path.exists(candidate):
            return candidate
    return os.path.join(base_parent, rel_path)

TRAIN_PATH = resolve_file(os.path.join("data", "processed", "train.csv"))
TEST_PATH = resolve_file(os.path.join("data", "processed", "test.csv"))
MODELS_DIR = os.path.join(base_parent, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Random Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment configured successfully.")
print(f"Train:  {TRAIN_PATH}")
print(f"Test:   {TEST_PATH}")
print(f"Models: {MODELS_DIR}")



Environment configured successfully.
Train:  ..\data\processed\train.csv
Test:   ..\data\processed\test.csv
Models: ..\models


---
## 1. Data Preparation (Sampling for Efficiency)
We load the training set. For Grid/Random Search, processing the entire dataset across dozens of parameter combinations can take hours. We will draw a randomized 50,000-row sample for tuning, while keeping the full test set for final evaluation.


In [2]:
# Load full training data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Define target variable
train_df['is_liked'] = (train_df['rating'] >= 4.0).astype(int)
test_df['is_liked'] = (test_df['rating'] >= 4.0).astype(int)

# Engineer features
user_avgs = train_df.groupby('userId')['rating'].mean().to_dict()
movie_avgs = train_df.groupby('movieId')['rating'].mean().to_dict()
global_avg = train_df['rating'].mean()

# Apply features to Train and Test
train_df['user_avg'] = train_df['userId'].map(user_avgs)
train_df['movie_avg'] = train_df['movieId'].map(movie_avgs)
test_df['user_avg'] = test_df['userId'].map(user_avgs).fillna(global_avg)
test_df['movie_avg'] = test_df['movieId'].map(movie_avgs).fillna(global_avg)

# Create a representative subsample for tuning (10,000 rows)
sample_train_df = train_df.sample(n=10000, random_state=RANDOM_STATE)

# Prepare Matrices
X_train_full = train_df[['user_avg', 'movie_avg']].values
y_train_full = train_df['is_liked'].values

X_train_sub = sample_train_df[['user_avg', 'movie_avg']].values
y_train_sub = sample_train_df['is_liked'].values

X_test = test_df[['user_avg', 'movie_avg']].values
y_test = test_df['is_liked'].values

print(f"Full Training Set: {X_train_full.shape}")
print(f"Tuning Subsample Set: {X_train_sub.shape}")
print(f"Test Set: {X_test.shape}")


Full Training Set: (826848, 2)
Tuning Subsample Set: (10000, 2)
Test Set: (206713, 2)


---
## 2. Train Advanced Models (Baselines)
We train `RandomForestClassifier` and `HistGradientBoostingClassifier` on the full training set using default parameters to see how they perform compared to the Week 3 Logistic Regression baseline.


In [3]:
# 1. Random Forest
rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=RANDOM_STATE)
robust_train_df = train_df.sample(n=20000, random_state=RANDOM_STATE)
X_train_robust = robust_train_df[['user_avg', 'movie_avg']].values
y_train_robust = robust_train_df['is_liked'].values

rf_model.fit(X_train_robust, y_train_robust)
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))
print(f"Random Forest (Default) Test Accuracy: {rf_test_acc * 100:.2f}%")

# 2. Histogram Gradient Boosting (Faster variant of XGBoost/LGBM natively in sklearn)
gb_model = HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE)
gb_model.fit(X_train_robust, y_train_robust)
gb_test_acc = accuracy_score(y_test, gb_model.predict(X_test))
print(f"Gradient Boosting (Default) Test Accuracy: {gb_test_acc * 100:.2f}%")


Random Forest (Default) Test Accuracy: 70.34%


Gradient Boosting (Default) Test Accuracy: 70.51%


---
## 3. K-Fold Cross-Validation
To ensure our model's performance is stable across different cuts of the data, we perform **5-Fold Stratified Cross-Validation**. We will use the `HistGradientBoostingClassifier` as it is highly efficient.


In [4]:
# Initialize Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Perform Cross-Validation on the tuning subsample
cv_scores = cross_val_score(
    HistGradientBoostingClassifier(random_state=RANDOM_STATE), 
    X_train_sub, 
    y_train_sub, 
    cv=cv, 
    scoring='accuracy'
)

print(f"Cross-Validation Scores (5 folds): {np.round(cv_scores, 4)}")
print(f"Mean CV Accuracy: {cv_scores.mean() * 100:.2f}% (Std: ±{cv_scores.std() * 100:.2f}%)")


Cross-Validation Scores (5 folds): [0.7055 0.6905 0.6995 0.7055 0.6945]
Mean CV Accuracy: 69.91% (Std: ±0.60%)


---
## 4. Hyperparameter Tuning (Randomized Search)
We optimize the Random Forest model by testing combinations of `max_depth`, `n_estimators`, and `min_samples_leaf` using `RandomizedSearchCV`.


In [5]:
# Define Hyperparameter Grid
param_distributions = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 5, 10]
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=param_distributions,
    n_iter=10,             # Number of random combinations to test
    cv=3,                  # 3-Fold CV for tuning
    scoring='accuracy',
    random_state=RANDOM_STATE,
    verbose=1
)

# Fit Randomized Search on the subsample
print("Starting Hyperparameter Tuning...")
random_search.fit(X_train_sub, y_train_sub)

# Results
print(f"\nBest Parameters Found: {random_search.best_params_}")
print(f"Best Tuning Accuracy:  {random_search.best_score_ * 100:.2f}%")

# Train the best model on a robust subsample (20000 rows) for maximum performance without memory issues
best_rf_model = random_search.best_estimator_
print("\nRetraining best model on robust dataset...")
best_rf_model.fit(X_train_robust, y_train_robust)

# Evaluate on the true Test Set
tuned_test_acc = accuracy_score(y_test, best_rf_model.predict(X_test))
print(f"Tuned Random Forest Test Accuracy: {tuned_test_acc * 100:.2f}%")

# Save the final tuned model
best_model_path = os.path.join(MODELS_DIR, "random_forest_tuned.joblib")
joblib.dump(best_rf_model, best_model_path)
print(f"\nTuned model saved to: {best_model_path}")


Starting Hyperparameter Tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits



Best Parameters Found: {'n_estimators': 150, 'min_samples_leaf': 10, 'max_depth': 5}
Best Tuning Accuracy:  71.09%

Retraining best model on robust dataset...


Tuned Random Forest Test Accuracy: 70.57%

Tuned model saved to: ..\models\random_forest_tuned.joblib


---
## ✅ Week 5 Completion Checklist

- [x] Implemented advanced non-linear models (Random Forest, Gradient Boosting).
- [x] Validated stability using 5-Fold Stratified Cross-Validation.
- [x] Defined hyperparameter grids for tuning.
- [x] Executed `RandomizedSearchCV` to find optimal `max_depth` and `n_estimators`.
- [x] Retrained the best model on the full training set and saved it.
